# Project 6: Quantum Generative Adversarial Network (QGAN)

## Introduction

**Generative Adversarial Networks (GANs)** are one of the most influential ideas in modern machine learning. Introduced by Goodfellow et al. (2014), a GAN consists of two neural networks locked in a competitive game:

- A **Generator** $G$ that tries to produce fake data indistinguishable from real data
- A **Discriminator** $D$ that tries to tell real data from fake

The two networks are trained simultaneously: the generator improves at fooling the discriminator, while the discriminator improves at detecting fakes. At equilibrium, the generator produces samples from the true data distribution.

### Quantum GANs

A **Quantum GAN (QGAN)** replaces the classical generator (and optionally the discriminator) with a parameterized quantum circuit. The quantum generator leverages the **Born machine** framework, where the output probability distribution is:

$$p_\theta(x) = |\langle x|U(\theta)|0\rangle|^2$$

Here $U(\theta)$ is a parameterized unitary acting on $n$ qubits, and $|x\rangle$ are computational basis states. The quantum circuit naturally produces a valid probability distribution over $2^n$ outcomes, making it an efficient generative model.

**Key advantages of QGANs:**
- Exponentially large output space with only $n$ qubits ($2^n$ probabilities)
- Natural normalization (Born rule guarantees valid distributions)
- Potential for loading complex distributions into quantum states for downstream quantum algorithms
- Applications in quantum finance, drug discovery, and quantum data augmentation

In this project, we will:
1. Build a hybrid QGAN with a quantum generator and classical discriminator
2. Train it to learn a target probability distribution
3. Implement a fully quantum GAN with a quantum discriminator
4. Apply QGANs to a financial use case (learning stock return distributions)

In [ ]:
# Imports
import pennylane as qml
from pennylane import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print(f"PennyLane version: {qml.__version__}")
print(f"PyTorch version: {torch.__version__}")

## Theory: GAN Training Dynamics

### Classical GAN Objective

The GAN training is formulated as a **minimax game**:

$$\min_G \max_D \; V(D, G) = \mathbb{E}_{x \sim p_{\text{data}}}[\log D(x)] + \mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))]$$

where:
- $p_{\text{data}}$ is the true data distribution
- $p_z$ is a noise prior (e.g., uniform or Gaussian)
- $G(z)$ maps noise to synthetic samples
- $D(x) \in [0, 1]$ outputs the probability that $x$ is real

### Quantum Generator

In a QGAN, the generator is a parameterized quantum circuit $U(\theta)$. Instead of mapping noise to samples, the quantum generator directly produces a probability distribution via the Born rule:

$$p_G(x; \theta) = |\langle x | U(\theta) | 0^n \rangle|^2$$

The generator's output is the full probability vector $\mathbf{p}_G = (p_G(0), p_G(1), \ldots, p_G(2^n - 1))$.

### Training the QGAN

The discriminator receives samples labeled as "real" (from $p_{\text{data}}$) or "fake" (from $p_G$) and outputs a probability of being real. The training alternates:

1. **Discriminator step**: Fix $\theta$, update $D$ to maximize $V(D, G)$
2. **Generator step**: Fix $D$, update $\theta$ to minimize $V(D, G)$

At convergence, $p_G(x; \theta^*) \approx p_{\text{data}}(x)$ and $D(x) \approx 0.5$ for all $x$.

### Convergence Metric: KL Divergence

We track learning progress using the Kullback-Leibler divergence:

$$D_{\text{KL}}(p_{\text{data}} \| p_G) = \sum_x p_{\text{data}}(x) \log \frac{p_{\text{data}}(x)}{p_G(x)}$$

## Target Distribution

We define a target distribution as a **mixture of Gaussians**, discretized to $2^3 = 8$ bins (3 qubits). This creates a non-trivial multi-modal distribution for the QGAN to learn.

In [ ]:
# Define the target distribution: Mixture of Gaussians discretized to 8 bins (3 qubits)
n_qubits = 3
n_bins = 2 ** n_qubits  # 8 bins

def mixture_of_gaussians(x, means, stds, weights):
    """Compute mixture of Gaussians PDF."""
    pdf = np.zeros_like(x)
    for mu, sigma, w in zip(means, stds, weights):
        pdf += w * np.exp(-0.5 * ((x - mu) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))
    return pdf

# Mixture parameters: two Gaussians
means = [2.0, 5.5]
stds = [0.8, 0.6]
weights = [0.4, 0.6]

# Discretize to 8 bins over [0, 8)
bin_edges = np.linspace(0, 8, n_bins + 1)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

# Compute target probabilities by integrating over each bin
target_probs = np.zeros(n_bins)
for i in range(n_bins):
    x_fine = np.linspace(bin_edges[i], bin_edges[i + 1], 100)
    target_probs[i] = np.trapz(mixture_of_gaussians(x_fine, means, stds, weights), x_fine)

# Normalize to ensure valid probability distribution
target_probs = target_probs / np.sum(target_probs)

# Convert to torch tensor
target_probs_torch = torch.tensor(target_probs, dtype=torch.float32)

print("Target distribution (8 bins):")
for i, p in enumerate(target_probs):
    print(f"  Bin {i} [{bin_edges[i]:.1f}, {bin_edges[i+1]:.1f}): p = {p:.4f}")
print(f"  Sum = {np.sum(target_probs):.6f}")

# Visualize the target distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Continuous PDF
x_cont = np.linspace(0, 8, 200)
ax1.plot(x_cont, mixture_of_gaussians(x_cont, means, stds, weights), 'b-', lw=2)
ax1.set_title('Continuous Mixture of Gaussians')
ax1.set_xlabel('x')
ax1.set_ylabel('PDF')
ax1.grid(True, alpha=0.3)

# Discretized distribution
ax2.bar(range(n_bins), target_probs, color='steelblue', alpha=0.8, edgecolor='navy')
ax2.set_title('Discretized Target Distribution (8 bins)')
ax2.set_xlabel('Bin index')
ax2.set_ylabel('Probability')
ax2.set_xticks(range(n_bins))
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Quantum Generator

The quantum generator is a parameterized quantum circuit acting on 3 qubits. We use **strongly entangling layers** from PennyLane, which consist of single-qubit rotations and CNOT entangling gates. The circuit outputs probabilities over the 8 computational basis states.

The generator circuit structure:
1. Initial layer of Hadamard gates to create superposition
2. Multiple layers of strongly entangling rotations
3. Measurement in the computational basis to obtain probabilities

In [ ]:
# Quantum Generator Circuit
n_gen_layers = 4

dev_gen = qml.device('default.qubit', wires=n_qubits)

@qml.qnode(dev_gen, interface='torch', diff_method='backprop')
def quantum_generator(params):
    """Parameterized quantum circuit as a generator.
    
    Args:
        params: Shape (n_layers, n_qubits, 3) - rotation angles
    
    Returns:
        Probabilities over 2^n_qubits outcomes
    """
    # Initial superposition
    for i in range(n_qubits):
        qml.Hadamard(wires=i)
    
    # Strongly entangling layers
    qml.StronglyEntanglingLayers(params, wires=range(n_qubits))
    
    return qml.probs(wires=range(n_qubits))

# Initialize generator parameters
gen_params = torch.randn(n_gen_layers, n_qubits, 3, requires_grad=True, dtype=torch.float32) * 0.5

# Test the generator
initial_probs = quantum_generator(gen_params)
print("Initial generator output probabilities:")
for i, p in enumerate(initial_probs.detach().numpy()):
    print(f"  |{i:03b}> = {p:.4f}")
print(f"  Sum = {initial_probs.sum().item():.6f}")

# Draw the circuit
print("\nGenerator circuit:")
print(qml.draw(quantum_generator, expansion_strategy='device')(gen_params))

## Classical Discriminator

The discriminator is a simple feedforward neural network that takes a probability value (for a given bin) and outputs the probability that it came from the real distribution. We use a small network since the problem is low-dimensional.

In [ ]:
class ClassicalDiscriminator(nn.Module):
    """Classical neural network discriminator.
    
    Takes a one-hot encoded bin index and outputs probability of being real.
    """
    def __init__(self, n_bins):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_bins, 32),
            nn.LeakyReLU(0.2),
            nn.Linear(32, 16),
            nn.LeakyReLU(0.2),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.net(x)

# Create discriminator
discriminator = ClassicalDiscriminator(n_bins)

# Test discriminator
test_input = torch.eye(n_bins)  # One-hot encoded bins
test_output = discriminator(test_input)
print("Discriminator test (one-hot inputs):")
for i in range(n_bins):
    print(f"  Bin {i}: D = {test_output[i].item():.4f}")

print(f"\nDiscriminator parameters: {sum(p.numel() for p in discriminator.parameters())}")

In [ ]:
# Training Loop: Hybrid QGAN (Quantum Generator + Classical Discriminator)

def kl_divergence(p, q, eps=1e-10):
    """Compute KL divergence D_KL(p || q)."""
    p_safe = torch.clamp(p, min=eps)
    q_safe = torch.clamp(q, min=eps)
    return torch.sum(p_safe * torch.log(p_safe / q_safe))

def sample_from_distribution(probs, n_samples):
    """Sample indices from a probability distribution and return one-hot encoded."""
    indices = torch.multinomial(probs, n_samples, replacement=True)
    one_hot = torch.zeros(n_samples, len(probs))
    one_hot.scatter_(1, indices.unsqueeze(1), 1.0)
    return one_hot

# Re-initialize models
gen_params = torch.randn(n_gen_layers, n_qubits, 3, requires_grad=True, dtype=torch.float32) * 0.5
discriminator = ClassicalDiscriminator(n_bins)

# Optimizers
gen_optimizer = optim.Adam([gen_params], lr=0.05)
disc_optimizer = optim.Adam(discriminator.parameters(), lr=0.01)

# Training parameters
n_iterations = 150
n_samples = 100  # Samples per batch
bce_loss = nn.BCELoss()

# Tracking metrics
gen_losses = []
disc_losses = []
kl_divs = []
prob_snapshots = []  # Store generator distributions at intervals
snapshot_iters = [0, 30, 60, 100, 149]

print("Training Hybrid QGAN...")
print("=" * 60)

for iteration in range(n_iterations):
    # --- Discriminator Training Step ---
    disc_optimizer.zero_grad()
    
    # Get generator probabilities
    gen_probs = quantum_generator(gen_params)
    
    # Sample from real and generated distributions
    real_samples = sample_from_distribution(target_probs_torch, n_samples)
    fake_samples = sample_from_distribution(gen_probs.detach(), n_samples)
    
    # Discriminator predictions
    real_pred = discriminator(real_samples)
    fake_pred = discriminator(fake_samples)
    
    # Discriminator loss: maximize log(D(real)) + log(1-D(fake))
    real_labels = torch.ones(n_samples, 1)
    fake_labels = torch.zeros(n_samples, 1)
    
    d_loss = bce_loss(real_pred, real_labels) + bce_loss(fake_pred, fake_labels)
    d_loss.backward()
    disc_optimizer.step()
    
    # --- Generator Training Step ---
    gen_optimizer.zero_grad()
    
    gen_probs = quantum_generator(gen_params)
    fake_samples = sample_from_distribution(gen_probs, n_samples)
    
    # Use straight-through estimator: weight discriminator output by probabilities
    # Alternative: directly optimize using the probability vector
    disc_on_gen = discriminator(torch.diag(gen_probs))
    g_loss = -torch.mean(torch.log(disc_on_gen + 1e-10))
    g_loss.backward()
    gen_optimizer.step()
    
    # Track metrics
    with torch.no_grad():
        current_probs = quantum_generator(gen_params)
        kl = kl_divergence(target_probs_torch, current_probs)
    
    gen_losses.append(g_loss.item())
    disc_losses.append(d_loss.item())
    kl_divs.append(kl.item())
    
    # Save snapshots
    if iteration in snapshot_iters:
        prob_snapshots.append(current_probs.detach().numpy().copy())
    
    if (iteration + 1) % 25 == 0:
        print(f"Iter {iteration+1:3d}/{n_iterations} | "
              f"D_loss: {d_loss.item():.4f} | G_loss: {g_loss.item():.4f} | "
              f"KL div: {kl.item():.4f}")

print("\nTraining complete!")

In [ ]:
# Visualization: Training Results

fig = plt.figure(figsize=(16, 12))
gs = GridSpec(3, 2, figure=fig, hspace=0.35, wspace=0.3)

# 1. Loss curves
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(gen_losses, label='Generator Loss', alpha=0.7, color='crimson')
ax1.plot(disc_losses, label='Discriminator Loss', alpha=0.7, color='royalblue')
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Loss')
ax1.set_title('GAN Training Losses')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. KL Divergence
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(kl_divs, color='darkgreen', lw=2)
ax2.set_xlabel('Iteration')
ax2.set_ylabel('KL Divergence')
ax2.set_title('KL Divergence (Target || Generated)')
ax2.set_yscale('log')
ax2.grid(True, alpha=0.3)

# 3. Distribution evolution snapshots
for idx, (snap_iter, snap_probs) in enumerate(zip(snapshot_iters, prob_snapshots)):
    row = 1 + idx // 2
    col = idx % 2
    if row < 3 or col < 2:  # Fit in the grid
        ax = fig.add_subplot(gs[row, col])
        width = 0.35
        x = np.arange(n_bins)
        ax.bar(x - width/2, target_probs, width, label='Target', color='steelblue', alpha=0.8)
        ax.bar(x + width/2, snap_probs, width, label='Generated', color='coral', alpha=0.8)
        ax.set_title(f'Iteration {snap_iter + 1}')
        ax.set_xlabel('Bin')
        ax.set_ylabel('Probability')
        ax.set_xticks(x)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Hybrid QGAN Training Progress', fontsize=14, fontweight='bold', y=1.01)
plt.show()

# Final comparison
fig, ax = plt.subplots(figsize=(10, 5))
final_probs = quantum_generator(gen_params).detach().numpy()
width = 0.35
x = np.arange(n_bins)
bars1 = ax.bar(x - width/2, target_probs, width, label='Target', color='steelblue', 
               alpha=0.85, edgecolor='navy')
bars2 = ax.bar(x + width/2, final_probs, width, label='QGAN Generated', color='coral', 
               alpha=0.85, edgecolor='darkred')
ax.set_xlabel('Bin Index', fontsize=12)
ax.set_ylabel('Probability', fontsize=12)
ax.set_title('Final Distribution Comparison', fontsize=14)
ax.set_xticks(x)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

final_kl = kl_divergence(target_probs_torch, torch.tensor(final_probs)).item()
ax.text(0.98, 0.95, f'KL Divergence = {final_kl:.4f}', transform=ax.transAxes,
        ha='right', va='top', fontsize=11, 
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
plt.tight_layout()
plt.show()

print("\nFinal distribution comparison:")
print(f"{'Bin':>4} {'Target':>10} {'Generated':>10} {'Abs Error':>10}")
print("-" * 40)
for i in range(n_bins):
    print(f"{i:>4} {target_probs[i]:>10.4f} {final_probs[i]:>10.4f} {abs(target_probs[i]-final_probs[i]):>10.4f}")
print(f"\nFinal KL Divergence: {final_kl:.6f}")

## Fully Quantum GAN

In a **fully quantum GAN**, both the generator and discriminator are parameterized quantum circuits. The quantum discriminator takes quantum states as input and outputs a measurement result indicating "real" or "fake".

Architecture:
- **Generator**: Same parameterized circuit producing states $|\psi_G\rangle = U_G(\theta_G)|0\rangle$
- **Discriminator**: A quantum circuit $U_D(\theta_D)$ that acts on the generator's output state, with a single-qubit measurement determining real/fake

The advantage of a fully quantum GAN is that it avoids the measurement bottleneck: the discriminator can process quantum states directly without collapsing them.

In [ ]:
# Fully Quantum GAN

n_disc_layers = 3

dev_full = qml.device('default.qubit', wires=n_qubits + 1)  # Extra ancilla for discriminator output

@qml.qnode(dev_full, interface='torch', diff_method='backprop')
def full_qgan_circuit(gen_params, disc_params, target_state=None):
    """Full quantum GAN circuit.
    
    Args:
        gen_params: Generator parameters (n_layers, n_qubits, 3)
        disc_params: Discriminator parameters (n_layers, n_qubits+1, 3)
        target_state: If provided, prepare target state instead of generator
    """
    gen_wires = list(range(n_qubits))
    all_wires = list(range(n_qubits + 1))
    
    if target_state is not None:
        # Prepare target/real state using amplitude embedding
        qml.AmplitudeEmbedding(target_state, wires=gen_wires, normalize=True)
    else:
        # Generator circuit
        for i in gen_wires:
            qml.Hadamard(wires=i)
        qml.StronglyEntanglingLayers(gen_params, wires=gen_wires)
    
    # Discriminator circuit (acts on all wires including ancilla)
    qml.StronglyEntanglingLayers(disc_params, wires=all_wires)
    
    # Measure ancilla qubit (wire n_qubits) as real/fake indicator
    return qml.expval(qml.PauliZ(n_qubits))

# Prepare target state (square root of probabilities)
target_amplitudes = torch.sqrt(target_probs_torch)

# Initialize parameters
fq_gen_params = torch.randn(n_gen_layers, n_qubits, 3, requires_grad=True, dtype=torch.float32) * 0.3
fq_disc_params = torch.randn(n_disc_layers, n_qubits + 1, 3, requires_grad=True, dtype=torch.float32) * 0.3

# Optimizers
fq_gen_opt = optim.Adam([fq_gen_params], lr=0.03)
fq_disc_opt = optim.Adam([fq_disc_params], lr=0.03)

# Training
fq_n_iterations = 150
fq_gen_losses = []
fq_disc_losses = []
fq_kl_divs = []

# Device for measuring generator probabilities
dev_measure = qml.device('default.qubit', wires=n_qubits)

@qml.qnode(dev_measure, interface='torch', diff_method='backprop')
def measure_gen_probs(params):
    for i in range(n_qubits):
        qml.Hadamard(wires=i)
    qml.StronglyEntanglingLayers(params, wires=range(n_qubits))
    return qml.probs(wires=range(n_qubits))

print("Training Fully Quantum GAN...")
print("=" * 60)

for iteration in range(fq_n_iterations):
    # --- Discriminator Step ---
    fq_disc_opt.zero_grad()
    
    # D should output +1 for real, -1 for fake
    real_output = full_qgan_circuit(fq_gen_params.detach(), fq_disc_params, 
                                    target_state=target_amplitudes)
    fake_output = full_qgan_circuit(fq_gen_params.detach(), fq_disc_params)
    
    # Discriminator wants to maximize: real_output - fake_output
    d_loss = -(real_output - fake_output)
    d_loss.backward()
    fq_disc_opt.step()
    
    # --- Generator Step ---
    fq_gen_opt.zero_grad()
    
    fake_output = full_qgan_circuit(fq_gen_params, fq_disc_params.detach())
    
    # Generator wants to maximize fake_output (fool discriminator)
    g_loss = -fake_output
    g_loss.backward()
    fq_gen_opt.step()
    
    # Track KL divergence
    with torch.no_grad():
        current_probs = measure_gen_probs(fq_gen_params)
        kl = kl_divergence(target_probs_torch, current_probs)
    
    fq_gen_losses.append(g_loss.item())
    fq_disc_losses.append(d_loss.item())
    fq_kl_divs.append(kl.item())
    
    if (iteration + 1) % 25 == 0:
        print(f"Iter {iteration+1:3d}/{fq_n_iterations} | "
              f"D_loss: {d_loss.item():.4f} | G_loss: {g_loss.item():.4f} | "
              f"KL div: {kl.item():.4f}")

print("\nTraining complete!")

# Compare both approaches
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# KL divergence comparison
axes[0].plot(kl_divs, label='Hybrid QGAN', color='royalblue', lw=2)
axes[0].plot(fq_kl_divs, label='Fully Quantum GAN', color='crimson', lw=2)
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('KL Divergence')
axes[0].set_title('Convergence Comparison')
axes[0].set_yscale('log')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Hybrid final distribution
hybrid_final = quantum_generator(gen_params).detach().numpy()
width = 0.3
x = np.arange(n_bins)
axes[1].bar(x - width/2, target_probs, width, label='Target', color='steelblue', alpha=0.8)
axes[1].bar(x + width/2, hybrid_final, width, label='Hybrid QGAN', color='coral', alpha=0.8)
axes[1].set_title('Hybrid QGAN Result')
axes[1].set_xlabel('Bin')
axes[1].set_ylabel('Probability')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

# Fully quantum final distribution
fq_final = measure_gen_probs(fq_gen_params).detach().numpy()
axes[2].bar(x - width/2, target_probs, width, label='Target', color='steelblue', alpha=0.8)
axes[2].bar(x + width/2, fq_final, width, label='Full QGAN', color='mediumpurple', alpha=0.8)
axes[2].set_title('Fully Quantum GAN Result')
axes[2].set_xlabel('Bin')
axes[2].set_ylabel('Probability')
axes[2].legend()
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"\nFinal KL Divergence - Hybrid: {kl_divs[-1]:.6f}, Fully Quantum: {fq_kl_divs[-1]:.6f}")

## Financial Application: Learning Stock Return Distributions

One of the most promising applications of QGANs is in **quantitative finance**. Financial institutions need to:
- Model complex probability distributions of asset returns
- Load these distributions into quantum states for downstream algorithms (e.g., quantum Monte Carlo, quantum risk analysis)

Here we train a QGAN to learn a **log-normal distribution** that models daily stock returns. The log-normal distribution is a standard model in finance (Black-Scholes framework):

$$f(x; \mu, \sigma) = \frac{1}{x\sigma\sqrt{2\pi}} \exp\left(-\frac{(\ln x - \mu)^2}{2\sigma^2}\right)$$

In [ ]:
# Financial Application: Log-normal Distribution for Stock Returns

# Parameters for log-normal distribution (typical daily stock returns)
mu_ln = 0.0005    # Daily expected return (~12% annually)
sigma_ln = 0.015  # Daily volatility (~24% annually)

# Generate synthetic stock return data
n_data = 10000
log_returns = np.random.normal(mu_ln, sigma_ln, n_data)
price_ratios = np.exp(log_returns)  # S(t+1)/S(t)

# Discretize to 8 bins
hist_range = (0.96, 1.04)  # Focus on typical return range
fin_bin_edges = np.linspace(hist_range[0], hist_range[1], n_bins + 1)
fin_bin_centers = (fin_bin_edges[:-1] + fin_bin_edges[1:]) / 2

# Compute empirical probabilities
counts, _ = np.histogram(price_ratios, bins=fin_bin_edges)
fin_target_probs = counts / counts.sum()
fin_target_torch = torch.tensor(fin_target_probs, dtype=torch.float32)

print("Financial target distribution (daily return ratios):")
for i in range(n_bins):
    pct = (fin_bin_centers[i] - 1) * 100
    print(f"  Bin {i}: [{fin_bin_edges[i]:.3f}, {fin_bin_edges[i+1]:.3f}) "
          f"= {pct:+.2f}%  p = {fin_target_probs[i]:.4f}")

# Train QGAN on financial distribution
fin_gen_params = torch.randn(n_gen_layers, n_qubits, 3, requires_grad=True, dtype=torch.float32) * 0.5
fin_disc = ClassicalDiscriminator(n_bins)

fin_gen_opt = optim.Adam([fin_gen_params], lr=0.05)
fin_disc_opt = optim.Adam(fin_disc.parameters(), lr=0.01)

fin_kl_divs = []

print("\nTraining QGAN on financial distribution...")
for iteration in range(150):
    # Discriminator step
    fin_disc_opt.zero_grad()
    gen_probs = quantum_generator(fin_gen_params)
    real_samples = sample_from_distribution(fin_target_torch, n_samples)
    fake_samples = sample_from_distribution(gen_probs.detach(), n_samples)
    real_pred = fin_disc(real_samples)
    fake_pred = fin_disc(fake_samples)
    d_loss = bce_loss(real_pred, torch.ones(n_samples, 1)) + bce_loss(fake_pred, torch.zeros(n_samples, 1))
    d_loss.backward()
    fin_disc_opt.step()
    
    # Generator step
    fin_gen_opt.zero_grad()
    gen_probs = quantum_generator(fin_gen_params)
    disc_on_gen = fin_disc(torch.diag(gen_probs))
    g_loss = -torch.mean(torch.log(disc_on_gen + 1e-10))
    g_loss.backward()
    fin_gen_opt.step()
    
    with torch.no_grad():
        current_probs = quantum_generator(fin_gen_params)
        kl = kl_divergence(fin_target_torch, current_probs)
    fin_kl_divs.append(kl.item())
    
    if (iteration + 1) % 50 == 0:
        print(f"  Iter {iteration+1}: KL = {kl.item():.4f}")

# Visualize financial results
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Original data histogram
axes[0].hist(price_ratios, bins=50, density=True, color='steelblue', alpha=0.7, edgecolor='navy')
axes[0].set_title('Simulated Daily Return Ratios')
axes[0].set_xlabel('S(t+1) / S(t)')
axes[0].set_ylabel('Density')
axes[0].axvline(x=1.0, color='red', linestyle='--', alpha=0.5, label='No change')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Discretized comparison
fin_final_probs = quantum_generator(fin_gen_params).detach().numpy()
width = 0.3
x = np.arange(n_bins)
axes[1].bar(x - width/2, fin_target_probs, width, label='Empirical', color='steelblue', alpha=0.8)
axes[1].bar(x + width/2, fin_final_probs, width, label='QGAN', color='coral', alpha=0.8)
axes[1].set_title('Distribution Comparison')
axes[1].set_xlabel('Bin')
axes[1].set_ylabel('Probability')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'{c:.3f}' for c in fin_bin_centers], rotation=45, fontsize=8)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

# KL convergence
axes[2].plot(fin_kl_divs, color='darkgreen', lw=2)
axes[2].set_title('Training Convergence')
axes[2].set_xlabel('Iteration')
axes[2].set_ylabel('KL Divergence')
axes[2].set_yscale('log')
axes[2].grid(True, alpha=0.3)

plt.suptitle('QGAN for Financial Distribution Learning', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nFinal KL Divergence (Financial): {fin_kl_divs[-1]:.6f}")

## Conclusion

In this project we explored **Quantum Generative Adversarial Networks** through three implementations:

### Key Results

1. **Hybrid QGAN** (Quantum Generator + Classical Discriminator):
   - Successfully learned a bimodal mixture of Gaussians distribution
   - The quantum generator (Born machine) naturally produces valid probability distributions
   - Training dynamics follow the classical GAN minimax game

2. **Fully Quantum GAN** (Quantum Generator + Quantum Discriminator):
   - Both generator and discriminator are parameterized quantum circuits
   - Avoids the measurement bottleneck by processing quantum states directly
   - Convergence behavior differs from the hybrid approach

3. **Financial Application**:
   - Demonstrated learning of log-normal stock return distributions
   - Practical relevance for quantum finance applications

### Advantages of QGANs
- **Exponential expressivity**: $n$ qubits represent $2^n$ probabilities
- **Built-in normalization**: Born rule guarantees valid distributions
- **Quantum state loading**: Learned distributions can be used in downstream quantum algorithms

### Limitations and Future Directions
- **Barren plateaus**: Deep quantum circuits may suffer from vanishing gradients
- **Noise**: Real quantum hardware introduces decoherence effects
- **Scalability**: Current demonstrations are limited to few qubits
- **Mode collapse**: Similar to classical GANs, QGANs can suffer from mode collapse

### References

1. Goodfellow, I. et al. "Generative Adversarial Nets." NeurIPS (2014).
2. Lloyd, S. & Weedbrook, C. "Quantum Generative Adversarial Learning." Phys. Rev. Lett. 121, 040502 (2018).
3. Dallaire-Demers, P. & Killoran, N. "Quantum generative adversarial networks." Phys. Rev. A 98, 012324 (2018).
4. Zoufal, C., Lucchi, A. & Woerner, S. "Quantum Generative Adversarial Networks for learning and loading random distributions." npj Quantum Information 5, 103 (2019).
5. Romero, J. & Aspuru-Guzik, A. "Variational quantum generators." arXiv:1901.00848 (2019).
6. Benedetti, M. et al. "Parameterized quantum circuits as machine learning models." Quantum Sci. Technol. 4, 043001 (2019).
7. Coyle, B. et al. "The Born Supremacy: Quantum Advantage and Training of an Ising Born Machine." npj Quantum Information 6, 60 (2020).